# Student-selected LightGBM minimum-leaf-size comparison

**12 October 2026:** the student explicitly chose **minimum leaf sizes 20 and 50**, against the existing setting of **5**. All three versions keep **150 trees**, learning rate 0.05, 15 leaves, 63 bins, minimum child Hessian weight 0.001, L2 penalty 1, seed 42, deterministic column-wise training and four CPU threads.

A leaf is the final prediction group in a decision tree. Raising `min_child_samples` asks LightGBM to avoid rules supported by very few examples. This may reduce memorization, but it may also miss a rare fraud pattern. Only this parameter changes. LightGBM's leaf-size restriction is an algorithmic constraint; the experiment does not assume a guarantee that every observed leaf count equals the requested minimum.

The student made this choice after the default-library comparison; the previously pending proposal is now explicitly selected. No shorter-training or other refinement is included. Keep all 165 original features and confirmed labels. Unknown rows do not receive invented labels and are excluded from this supervised comparison.

Keep the same 15 chronological development checks: train through k=7,…,21 and check the next 14 steps. Freshly fit the unchanged current ensemble recipe as a control using the 5-example LightGBM version plus its filtered low-regularisation XGBoost component. Its feature filter uses fitting-period rows only. Also compare directly with the standalone 5-example LightGBM. Show all-15, earlier-eight, later-seven, previously chosen four and latest-check results; reused overlapping checks are not independent held-out tests or public-score forecasts.

The submitted ensemble's known public ROC-AUC is 0.94701. Public scores for these standalone candidates are unknown. Three candidate settings are declared before training; no outer checking labels choose stopping, alter parameters or train a model. No blend search, automatic final selection or Kaggle upload occurs.

**AI assistance:** the student selected the fixed leaf-size comparison and existing validation design. OpenAI Codex implemented and ran it, verified results and drafted the factual record. The student remains responsible for interpretation, explanation, final selection and disclosure.

Run all cells in a fresh CPU Python kernel. Set DATA_DIR and OUTPUT_DIR if needed. This notebook generates predictions using only the supplied train/test/sample files and recorded software libraries; no prior model, prediction file or experiment notebook is required.


In [1]:
from pathlib import Path
import os, json, hashlib, time, platform
for variable in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[variable] = "4"
import numpy as np
import pandas as pd
import sklearn, xgboost, lightgbm
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier, Booster
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr
from threadpoolctl import threadpool_limits

DATA_DIR = Path(os.environ.get("BT4012_DATA_DIR", "/Users/mlvlo/BT4012/Kaggle/Data Set"))
if not DATA_DIR.exists():
    DATA_DIR = Path("/kaggle/input") if Path("/kaggle/input").exists() else Path.cwd()
OUTPUT_DIR = Path(os.environ.get("BT4012_LEAF_OUTPUT_DIR", str(Path.cwd() / "lightgbm_leaf_results")))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FEATURES = [f"feat_{i}" for i in range(1, 166)]
LEAF_SIZES = (5, 20, 50)
CHECKS = tuple((end, end + 1, end + 14) for end in range(7, 22))
SELECTED_FOUR = (14, 17, 19, 21)
SEED = 42
STARTED = time.perf_counter()
FIXED_LGBM = dict(n_estimators=150, learning_rate=0.05, num_leaves=15, max_bin=63,
                 min_child_weight=0.001, reg_lambda=1,
                 objective="binary", deterministic=True, force_col_wise=True,
                 verbosity=-1, n_jobs=4, random_state=SEED)
RECIPE = dict(min_child_samples_values=list(LEAF_SIZES), fixed_lightgbm=FIXED_LGBM,
              chronological_checks=[list(check) for check in CHECKS],
              selected_four=list(SELECTED_FOUR), existing_public_auc=0.94701,
              student_decision="Test minimum leaf sizes of 20 and 50 (Recommended)",
              final_selection="pending student review", kaggle_upload=False)
recipe_file = OUTPUT_DIR / "declared_recipe.json"
if recipe_file.exists():
    assert json.loads(recipe_file.read_text()) == RECIPE, "Different prior recipe: use a new output folder."
else:
    recipe_file.write_text(json.dumps(RECIPE, indent=2))

def digest(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def provided_file(name):
    matches = sorted(DATA_DIR.rglob(name))
    assert len(matches) == 1, f"Expected exactly one provided {name}: found {matches}"
    return matches[0]

def progress(phase, **details):
    (OUTPUT_DIR / "progress.json").write_text(json.dumps(dict(
        phase=phase, elapsed_seconds=round(time.perf_counter() - STARTED, 3), **details), indent=2))

print("Declared minimum leaf sizes:", LEAF_SIZES, "; fixed settings:", FIXED_LGBM)
print("Results folder:", OUTPUT_DIR)

Declared minimum leaf sizes: (5, 20, 50) ; fixed settings: {'n_estimators': 150, 'learning_rate': 0.05, 'num_leaves': 15, 'max_bin': 63, 'min_child_weight': 0.001, 'reg_lambda': 1, 'objective': 'binary', 'deterministic': True, 'force_col_wise': True, 'verbosity': -1, 'n_jobs': 4, 'random_state': 42}
Results folder: /Users/mlvlo/.codex/.chatgpt-projects/g-p-6a901a23216481918fe5444e40decc24/student_model_review/lightgbm_leaf_results


In [2]:
train_path = provided_file("Train Data.csv")
test_path = provided_file("Test Data.csv")
sample_path = provided_file("BT4012 Competition Submission.csv")
SOURCE_HASHES = {str(p): digest(p) for p in (train_path, test_path, sample_path)}
train = pd.read_csv(train_path)
numeric_labels = pd.to_numeric(train["label"], errors="coerce")
known = train.loc[numeric_labels.notna()].copy()
known["label"] = numeric_labels.loc[numeric_labels.notna()].astype(int)
test = pd.read_csv(test_path)
sample = pd.read_csv(sample_path)
assert numeric_labels.dropna().isin([0, 1]).all()
assert known["label"].nunique() == 2
assert train["txId"].is_unique and test["txId"].is_unique
assert set(train["txId"]).isdisjoint(test["txId"])
assert train["time_step"].max() < test["time_step"].min()
assert test["index"].is_unique and sample["index"].is_unique
assert set(test["index"]) == set(sample["index"])
assert sample.columns.tolist() == ["index", "target"]
for rows in (train, test):
    assert np.isfinite(rows[FEATURES].to_numpy()).all()

def make_lightgbm(minimum):
    assert minimum in LEAF_SIZES
    return LGBMClassifier(min_child_samples=minimum, **FIXED_LGBM)

def existing_xgb_columns(learning):
    # Preserve the submitted ensemble's filter, using fitting data only.
    cutoff = int(learning["time_step"].max())
    filter_rows = learning.loc[learning["time_step"] <= cutoff - 3]
    correlations = filter_rows[FEATURES].corrwith(filter_rows["time_step"], method="spearman")
    removed = correlations.index[correlations.abs() > 0.9].tolist()
    return [column for column in FEATURES if column not in removed], removed

def make_existing_xgb():
    return XGBClassifier(n_estimators=200, learning_rate=0.05, tree_method="hist",
                         grow_policy="lossguide", max_depth=0, max_leaves=15,
                         max_bin=127, min_child_weight=1, reg_lambda=1,
                         objective="binary:logistic", n_jobs=4, random_state=SEED)

print(f"Training rows: {len(train):,}; confirmed labels: {len(known):,}; "
      f"unknown rows: {int(numeric_labels.isna().sum()):,}; test rows: {len(test):,}")
print("Confirmed labels:", known["label"].value_counts().to_dict())

Training rows: 141,772; confirmed labels: 31,235; unknown rows: 110,537; test rows: 15,329
Confirmed labels: {0: 27591, 1: 3644}


In [3]:
results_path = OUTPUT_DIR / "validation_results.csv"
arrays_path = OUTPUT_DIR / "validation_probabilities.npz"
records, arrays, selections = [], {}, {}
if results_path.exists():
    assert arrays_path.exists()
    records = pd.read_csv(results_path).to_dict(orient="records")
    with np.load(arrays_path) as saved: arrays = {key: saved[key] for key in saved.files}
    assert len(records) == len({int(row['train_end']) for row in records})
for cutoff, first, last in CHECKS:
    learning = known.loc[known.time_step <= cutoff]
    checking = known.loc[known.time_step.between(first, last)]
    assert learning.time_step.max() < checking.time_step.min()
    assert set(learning.txId).isdisjoint(checking.txId)
    assert learning.label.nunique() == checking.label.nunique() == 2
    columns, removed = existing_xgb_columns(learning)
    selections[str(cutoff)] = removed
    if any(int(row['train_end']) == cutoff for row in records):
        assert np.array_equal(arrays[f'labels_{cutoff}'], checking.label.to_numpy())
        assert np.array_equal(arrays[f'txIds_{cutoff}'], checking.txId.to_numpy())
        continue
    progress('chronological training', cutoff=cutoff, completed_checks=len(records), total_checks=15)
    tick = time.perf_counter()
    scores = {}
    with threadpool_limits(limits=4):
        for minimum in LEAF_SIZES:
            model = make_lightgbm(minimum)
            model.fit(learning[FEATURES], learning.label)
            scores[f'lightgbm_leaf_{minimum}'] = model.predict_proba(checking[FEATURES])[:, 1]
            assert model.n_estimators_ == 150
            assert model.get_params()['min_child_samples'] == minimum
        control_xgb = make_existing_xgb()
        control_xgb.fit(learning[columns], learning.label)
        p_xgb = control_xgb.predict_proba(checking[columns])[:, 1]
    scores['current_pair'] = 0.5 * p_xgb + 0.5 * scores['lightgbm_leaf_5']
    labels = checking.label.to_numpy()
    row = dict(train_end=cutoff, validation_start=first, validation_end=last,
               learning_rows=len(learning), checking_rows=len(checking),
               learning_illicit=int(learning.label.sum()), checking_illicit=int(labels.sum()),
               elapsed_seconds=round(time.perf_counter() - tick, 3))
    for name, probabilities in scores.items():
        assert np.isfinite(probabilities).all() and ((probabilities >= 0) & (probabilities <= 1)).all()
        row[name + '_auc'] = float(roc_auc_score(labels, probabilities))
        arrays[f'{name}_{cutoff}'] = probabilities
    for minimum in LEAF_SIZES:
        row[f'lightgbm_leaf_{minimum}_minus_pair'] = row[f'lightgbm_leaf_{minimum}_auc'] - row['current_pair_auc']
        row[f'lightgbm_leaf_{minimum}_minus_5'] = row[f'lightgbm_leaf_{minimum}_auc'] - row['lightgbm_leaf_5_auc']
    arrays[f'labels_{cutoff}'] = labels
    arrays[f'txIds_{cutoff}'] = checking.txId.to_numpy()
    arrays[f'time_steps_{cutoff}'] = checking.time_step.to_numpy()
    records.append(row)
    np.savez_compressed(arrays_path, **arrays)
    pd.DataFrame(records).to_csv(results_path, index=False)
    print(f"Cutoff {cutoff:2}: pair {row['current_pair_auc']:.6f}; " + '; '.join(
        f"leaf minimum {minimum}: {row[f'lightgbm_leaf_{minimum}_auc']:.6f}" for minimum in LEAF_SIZES))
validation = pd.DataFrame(records).sort_values('train_end').reset_index(drop=True)
assert validation.train_end.tolist() == list(range(7, 22))
validation.to_csv(results_path, index=False)
(OUTPUT_DIR / 'xgboost_removed_features.json').write_text(json.dumps(selections, indent=2))
progress('all chronological checks complete', completed_checks=15)
display(validation[['train_end', 'current_pair_auc'] + [f'lightgbm_leaf_{n}_auc' for n in LEAF_SIZES]])

Cutoff  7: pair 0.992985; leaf minimum 5: 0.991691; leaf minimum 20: 0.992501; leaf minimum 50: 0.992588
Cutoff  8: pair 0.993827; leaf minimum 5: 0.993200; leaf minimum 20: 0.993325; leaf minimum 50: 0.993597
Cutoff  9: pair 0.991915; leaf minimum 5: 0.990934; leaf minimum 20: 0.991651; leaf minimum 50: 0.991773
Cutoff 10: pair 0.977749; leaf minimum 5: 0.976330; leaf minimum 20: 0.977907; leaf minimum 50: 0.975622
Cutoff 11: pair 0.975593; leaf minimum 5: 0.973871; leaf minimum 20: 0.973723; leaf minimum 50: 0.971731
Cutoff 12: pair 0.969104; leaf minimum 5: 0.968787; leaf minimum 20: 0.973822; leaf minimum 50: 0.978432
Cutoff 13: pair 0.971758; leaf minimum 5: 0.969207; leaf minimum 20: 0.969327; leaf minimum 50: 0.976040
Cutoff 14: pair 0.972177; leaf minimum 5: 0.968360; leaf minimum 20: 0.969933; leaf minimum 50: 0.978591
Cutoff 15: pair 0.939024; leaf minimum 5: 0.943194; leaf minimum 20: 0.946718; leaf minimum 50: 0.947238
Cutoff 16: pair 0.947183; leaf minimum 5: 0.948616; lea

,train_end,current_pair_auc,lightgbm_leaf_5_auc,lightgbm_leaf_20_auc,lightgbm_leaf_50_auc
0,7,0.992985,0.991691,0.992501,0.992588
1,8,0.993827,0.993200,0.993325,0.993597
2,9,0.991915,0.990934,0.991651,0.991773
3,10,0.977749,0.976330,0.977907,0.975622
4,11,0.975593,0.973871,0.973723,0.971731
5,12,0.969104,0.968787,0.973822,0.978432
6,13,0.971758,0.969207,0.969327,0.976040
7,14,0.972177,0.968360,0.969933,0.978591
8,15,0.939024,0.943194,0.946718,0.947238
9,16,0.947183,0.948616,0.942435,0.946034


In [4]:
VARIANTS = ('current_pair',) + tuple(f'lightgbm_leaf_{minimum}' for minimum in LEAF_SIZES)
GROUPS = {'all_15': validation, 'earlier_eight': validation.loc[validation.train_end <= 14],
          'later_seven': validation.loc[validation.train_end >= 15],
          'selected_four': validation.loc[validation.train_end.isin(SELECTED_FOUR)],
          'latest': validation.loc[validation.train_end == 21]}
summary_rows = []
for group, rows in GROUPS.items():
    for variant in VARIANTS:
        values = rows[variant + '_auc']
        summary_rows.append(dict(group=group, model=variant, checks=len(rows), mean_auc=float(values.mean()),
            worst_auc=float(values.min()), mean_delta_vs_pair=float((values - rows.current_pair_auc).mean()),
            wins_vs_pair=int((values > rows.current_pair_auc).sum()),
            mean_delta_vs_leaf5=float((values - rows.lightgbm_leaf_5_auc).mean()),
            wins_vs_leaf5=int((values > rows.lightgbm_leaf_5_auc).sum())))
summary = pd.DataFrame(summary_rows)
summary.to_csv(OUTPUT_DIR / 'summary.csv', index=False)
per_period = []
for cutoff, _, _ in CHECKS:
    periods, labels = arrays[f'time_steps_{cutoff}'], arrays[f'labels_{cutoff}']
    for step in np.unique(periods):
        mask = periods == step
        row = dict(train_end=cutoff, time_step=int(step), rows=int(mask.sum()), illicit=int(labels[mask].sum()))
        for variant in VARIANTS:
            row[variant + '_auc'] = float(roc_auc_score(labels[mask], arrays[f'{variant}_{cutoff}'][mask])) if len(np.unique(labels[mask])) == 2 else np.nan
        per_period.append(row)
pd.DataFrame(per_period).to_csv(OUTPUT_DIR / 'per_time_step.csv', index=False)
checking_ids = np.concatenate([arrays[f'txIds_{end}'] for end, _, _ in CHECKS])
unique_ids, appearances = np.unique(checking_ids, return_counts=True)
display(summary)
print(f'{len(checking_ids):,} checking-row appearances cover {len(unique_ids):,} unique transactions, up to {int(appearances.max())} appearances each.')
print('Public scores remain unknown. Final selection stays with the student.')

163,794 checking-row appearances cover 21,682 unique transactions, up to 14 appearances each.
Public scores remain unknown. Final selection stays with the student.


,group,model,checks,mean_auc,worst_auc,mean_delta_vs_pair,wins_vs_pair,mean_delta_vs_leaf5,wins_vs_leaf5
0,all_15,current_pair,15,0.954972,0.907205,0.000000,0,-0.000761,8
1,all_15,lightgbm_leaf_5,15,0.955732,0.908792,0.000761,7,0.000000,0
2,all_15,lightgbm_leaf_20,15,0.955253,0.898663,0.000282,6,-0.000479,9
3,all_15,lightgbm_leaf_50,15,0.958422,0.905247,0.003451,7,0.002690,10
4,earlier_eight,current_pair,8,0.980638,0.969104,0.000000,0,0.001591,8
5,earlier_eight,lightgbm_leaf_5,8,0.979047,0.968360,-0.001591,0,0.000000,0
6,earlier_eight,lightgbm_leaf_20,8,0.980274,0.969327,-0.000365,2,0.001226,7
7,earlier_eight,lightgbm_leaf_50,8,0.982297,0.971731,0.001658,3,0.003249,6
8,later_seven,current_pair,7,0.925638,0.907205,0.000000,0,-0.003448,0
9,later_seven,lightgbm_leaf_5,7,0.929086,0.908792,0.003448,7,0.000000,0


In [5]:
final_rows, full_predictions = [], {}
for minimum in LEAF_SIZES:
    progress('full training and artifact checks', min_child_samples=minimum)
    model = make_lightgbm(minimum)
    with threadpool_limits(limits=4):
        model.fit(known[FEATURES], known.label)
        probabilities = np.asarray(model.predict_proba(test[FEATURES])[:, 1], dtype=np.float64)
    assert model.n_estimators_ == 150 and model.get_params()['min_child_samples'] == minimum
    aligned = sample['index'].map(pd.Series(probabilities, index=test['index'])).to_numpy()
    assert len(aligned) == len(sample) and np.isfinite(aligned).all() and ((aligned >= 0) & (aligned <= 1)).all()
    model_path = OUTPUT_DIR / f'lightgbm_leaf_{minimum}_model.txt'
    model.booster_.save_model(str(model_path))
    with threadpool_limits(limits=4):
        restored = Booster(model_file=str(model_path))
        restored_probabilities = restored.predict(test[FEATURES], num_threads=4)
    assert restored.feature_name() == FEATURES and restored.num_trees() == 150
    reload_error = float(np.max(np.abs(restored_probabilities - probabilities)))
    assert reload_error <= 1e-15
    csv_path = OUTPUT_DIR / f'submission_lightgbm_leaf_{minimum}_experimental.csv'
    pd.DataFrame({'index': sample['index'], 'target': aligned}).to_csv(csv_path, index=False)
    reread = pd.read_csv(csv_path)
    assert reread.columns.tolist() == ['index', 'target'] and reread['index'].equals(sample['index'])
    assert np.isfinite(reread.target).all() and reread.target.between(0, 1).all()
    assert np.allclose(reread.target, aligned, rtol=0, atol=1e-15)
    full_predictions[f'lightgbm_leaf_{minimum}'] = aligned
    final_rows.append(dict(model=f'lightgbm_leaf_{minimum}', min_child_samples=minimum, trees=150,
        training_rows=len(known), training_illicit=int(known.label.sum()), test_rows=len(sample),
        unknown_training_rows_excluded=int(numeric_labels.isna().sum()), saved_model_reload_max_error=reload_error,
        submission_sha256=digest(csv_path), model_sha256=digest(model_path)))
    print(f'Saved minimum-leaf-size {minimum} model and verified {len(sample):,} prediction rows.')
full_predictions['indices'] = sample['index'].to_numpy()
np.savez_compressed(OUTPUT_DIR / 'test_probabilities.npz', **full_predictions)
final_training = pd.DataFrame(final_rows)
final_training.to_csv(OUTPUT_DIR / 'final_training.csv', index=False)
metadata = dict(RECIPE, source_sha256=SOURCE_HASHES, train_rows=len(train), known_training_rows=len(known),
    unknown_training_rows=int(numeric_labels.isna().sum()), test_rows=len(test),
    unique_checking_transactions=len(unique_ids), checking_row_appearances=len(checking_ids),
    maximum_checking_appearances=int(appearances.max()), future_rows_used_for_fitting=False,
    unknown_rows_given_labels=False, all_saved_models_verified=True, outer_auc_used_for_stopping=False,
    original_data_unchanged=all(digest(path) == value for path, value in SOURCE_HASHES.items()),
    python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__, sklearn=sklearn.__version__,
    lightgbm=lightgbm.__version__, xgboost=xgboost.__version__, elapsed_seconds=round(time.perf_counter() - STARTED, 3))
assert metadata['original_data_unchanged']
(OUTPUT_DIR / 'experiment_metadata.json').write_text(json.dumps(metadata, indent=2))
progress('complete', completed_checks=15, full_models=3)
display(final_training)
print('Completed the fixed leaf-size experiment. No further tuning, final selection or Kaggle upload.')

Saved minimum-leaf-size 5 model and verified 15,329 prediction rows.
Saved minimum-leaf-size 20 model and verified 15,329 prediction rows.
Saved minimum-leaf-size 50 model and verified 15,329 prediction rows.
Completed the fixed leaf-size experiment. No further tuning, final selection or Kaggle upload.


,model,min_child_samples,trees,training_rows,training_illicit,test_rows,unknown_training_rows_excluded,saved_model_reload_max_error,submission_sha256,model_sha256
0,lightgbm_leaf_5,5,150,31235,3644,15329,110537,0.0,fb7545a1c1c7269de724d55430217d55ec912ff86fa5db...,2ff9771b75ab03f1260d36815851dfc2022e07c2afaee3...
1,lightgbm_leaf_20,20,150,31235,3644,15329,110537,0.0,6ebb20ad9150042ea15c43e163a585b2e9678e77ca15ba...,6d5921048c7ba54e860df8e6ad5195b85331fcb04ff41b...
2,lightgbm_leaf_50,50,150,31235,3644,15329,110537,0.0,93eae6e7ffc4ef0bae305f31cf4e107072d97875747598...,b6e4ce9553aea144936b2c86fd22e4d0f2ab46f3ab9fd7...


## Measured findings

The fixed comparison completed all **15 chronological checks** with fresh unchanged controls and full fits of the three declared candidates. Every full fit used **31,235 confirmed labels** through step 35 and generated **15,329 verified test predictions**. **110,537 unknown rows** received no invented labels and were excluded from these supervised fits. Experiment elapsed time: **40.6 seconds**, four CPU threads.

| Check group | Submitted ensemble recipe | LightGBM leaf 5 | LightGBM leaf 20 | LightGBM leaf 50 |
|---|---:|---:|---:|---:|
| All 15 | 0.954972 | 0.955732 | 0.955253 | 0.958422 |
| Earlier eight | 0.980638 | 0.979047 | 0.980274 | 0.982297 |
| Later seven | 0.925638 | 0.929086 | 0.926659 | 0.931137 |
| Previously chosen four | 0.935353 | 0.937024 | 0.937178 | 0.942234 |
| Latest: train 1–21, check 22–35 | 0.916956 | 0.919396 | 0.918323 | 0.916709 |

- **Minimum 20:** change against leaf 5 is -0.000479 AUC over all 15, -0.002428 over the later seven, and -0.001073 on the latest check. Wins 9/15 and 2/7 later checks.
- **Minimum 50:** change against leaf 5 is +0.002690 AUC over all 15, +0.002051 over the later seven, and -0.002687 on the latest check. Wins 10/15 and 4/7 later checks.

**AI-assisted recommendation:** The 50-example version merits review as a candidate because it has the highest later-seven mean among the three. The highest all-15 mean occurs at 50 examples, while the latest-check leader is 5. Do not treat the mean improvement as proof of a better Kaggle score; review the latest and worst-check declines before selecting a submission. Public scores for the standalone candidates remain unknown; the screenshot-confirmed public high score remains **0.94701**. No forecast of 0.96+ has been established.

### Interpretation and tradeoffs

Leaf 50's higher mean supports investigating less detailed prediction rules at this fixed training length. It does not establish the cause of every improvement or prove that every larger minimum will work. Leaf 20 performs worse on the later-seven mean than leaf 5, so the relationship is not monotonic. All candidate settings were declared before looking at checking results; no setting or stopping point was revised after seeing outer scores.

The leaf-50 version's worst fourteen-period AUC is **0.905247**, compared with **0.908792** for leaf 5. Its latest-check AUC is **0.916709**, compared with **0.919396** for leaf 5 and **0.916956** for the submitted ensemble recipe. Those declines are material even though mean performance improved. Both old and new candidates remain available for student review; no model has automatically replaced a prior submission.

Only min_child_samples changed. This parameter asks LightGBM to restrict very small terminal groups; it does not remove the overall 15-leaf limit or change the 150-tree budget. A larger minimum can smooth predictions but can also combine examples that need different fraud rankings. The measured scores are the evidence for this specific parameter choice. The experiment does not label unknown transactions or fit future features.

The later checks involve more recent periods, but they overlap the earlier checks and have been reused during development. **163,794 checking-row appearances cover 21,682 unique transactions**, with up to **14 appearances each. Means, win counts and differences are development evidence, not independent held-out proof or calibrated public-score expectations. Continued comparison across these same windows can lead to selection optimism.

### Verification and reproducibility

All five code cells executed with genuine captured outputs. Independent verification recalculated every AUC and summary, checked identifiers, labels and fitting counts, and confirmed the settings. The leaf-5 probabilities exactly reproduced the previous 150-tree LightGBM on every validation check **and all full test predictions**. The freshly fitted ensemble control also reproduced every previous validation probability exactly. This establishes a matched comparison with the prior experiments.

All three saved LightGBM text models reproduced test predictions after reload to within 1e−15. All three CSVs passed row-count, column, identifier/order, finite-probability, range and round-trip checks; original data hashes remained unchanged. The notebook is standalone and generates predictions using only supplied competition data and recorded libraries, without prior prediction arrays. Prediction generation used LightGBM **4.7.0**, XGBoost **3.2.0**, scikit-learn **1.9.1**, NumPy **2.4.6** and pandas **2.3.3**.

Complete source and genuine outputs: `e1156285_lightgbm_leaf.ipynb`. Declared recipe, checking probabilities, tables, saved models, experimental prediction CSVs and independent verification: `lightgbm_leaf_results/`.

**Student ownership and AI disclosure:** on 12 October 2026, the student explicitly chose the minimum-leaf-size 20/50 comparison against 5. OpenAI Codex implemented and ran the chosen experiment, verified artifacts and drafted this record. The student is responsible for understanding and interpreting results and deciding what to submit, and must disclose assistance accurately. No shorter-training comparison, unselected next experiment, automatic blend search, final-model selection or Kaggle upload occurred.

### Every chronological result

| Train through | Check steps | Ensemble | Leaf 5 | Leaf 20 | Leaf 50 |
|---:|---|---:|---:|---:|---:|
| 7 | 8–21 | 0.992985 | 0.991691 | 0.992501 | 0.992588 |
| 8 | 9–22 | 0.993827 | 0.993200 | 0.993325 | 0.993597 |
| 9 | 10–23 | 0.991915 | 0.990934 | 0.991651 | 0.991773 |
| 10 | 11–24 | 0.977749 | 0.976330 | 0.977907 | 0.975622 |
| 11 | 12–25 | 0.975593 | 0.973871 | 0.973723 | 0.971731 |
| 12 | 13–26 | 0.969104 | 0.968787 | 0.973822 | 0.978432 |
| 13 | 14–27 | 0.971758 | 0.969207 | 0.969327 | 0.976040 |
| 14 | 15–28 | 0.972177 | 0.968360 | 0.969933 | 0.978591 |
| 15 | 16–29 | 0.939024 | 0.943194 | 0.946718 | 0.947238 |
| 16 | 17–30 | 0.947183 | 0.948616 | 0.942435 | 0.946034 |
| 17 | 18–31 | 0.939401 | 0.941784 | 0.937318 | 0.946074 |
| 18 | 19–32 | 0.916818 | 0.923267 | 0.920014 | 0.929097 |
| 19 | 20–33 | 0.912879 | 0.918555 | 0.923139 | 0.927562 |
| 20 | 21–34 | 0.907205 | 0.908792 | 0.898663 | 0.905247 |
| 21 | 22–35 | 0.916956 | 0.919396 | 0.918323 | 0.916709 |
